# Explainability - part 1: SHAP vs. an interpretable model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomasvicar/AUI-public/blob/master/labs/explainability/notebooks/ex3_shap.ipynb)

**Computer-Aided Medical Diagnostics (MPA-PRM)** · lab for the lecture
*Explainability and interpretability of models*

Two models on the same 303 patients: a logistic regression, interpretable by
design, and an XGBoost black box explained after the fact with SHAP. Is the
black box worth it - and do the two explanations agree?

**Six holes.** The logistic regression and its coefficients, the XGBoost,
the global and the local SHAP views, and the comparison.

In [ ]:
%pip install -q shap

# Data

UCI Heart Disease, Cleveland (CC BY 4.0), stored in the course repository.
The target `num` is graded 0-4; the lab uses *any disease* (`num > 0`). The
split is 75 / 25 %, stratified, seed 42.

In [ ]:
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

URL = "https://raw.githubusercontent.com/tomasvicar/AUI-public/master/labs/robustness/data/heart_disease_cleveland.csv"
CSV = Path("heart_disease_cleveland.csv")
if not CSV.is_file():
    urllib.request.urlretrieve(URL, CSV)

SEED = 42
FEATURE_NAMES = {
    "age": "age", "sex": "sex (1 = male)", "cp": "chest pain type", "trestbps": "resting blood pressure",
    "chol": "cholesterol", "fbs": "fasting blood sugar > 120", "restecg": "resting ECG",
    "thalach": "max heart rate", "exang": "exercise angina", "oldpeak": "ST depression",
    "slope": "ST slope", "ca": "vessels on fluoroscopy", "thal": "thallium test",
}


def load_heart(path: Path = CSV) -> tuple[pd.DataFrame, pd.Series]:
    """13 features (missing values filled with the median) and disease = num > 0."""
    table = pd.read_csv(path)
    y = (table.pop("num") > 0).astype(int)
    return table.fillna(table.median(numeric_only=True)), y


def split(X: pd.DataFrame, y: pd.Series, seed: int = SEED):
    """Train / test = 75 / 25 %, stratified."""
    return train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)


X, y = load_heart()
X_train, X_test, y_train, y_test = split(X, y)
print(f"{len(X)} patients, {X.shape[1]} features, {y.mean():.0%} with the disease")
print(f"train {len(X_train)}, test {len(X_test)}")
X.head()

# The interpretable model

### Hole 1 - logistic regression

In [ ]:
# TODO 1: logistic regression on STANDARDIZED features.
#
# make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED))
# fitted on the training part; then the probability of disease on the test
# part, p_logistic = logistic.predict_proba(X_test)[:, 1], the accuracy of
# the prediction p_logistic > 0.5 and the AUC (roc_auc_score).

logistic = ...

### Hole 2 - its coefficients are its explanation

Standardized features: one standard deviation of each feature changes the
log-odds by its coefficient. Which features have the largest effect - and
does the sign match your clinical intuition?

In [ ]:
# TODO 2: the model IS its explanation. Take the coefficients of the logistic
#         regression (logistic[-1].coef_[0], one per standardized feature),
#         sort them and draw them as horizontal bars - red for the ones that
#         push towards disease, blue for the protective ones.

coefficients = ...

# The black box

### Hole 3 - XGBoost

Before you run it: write down how much better you expect it to be.

In [ ]:
# TODO 3: the black box - the shared recipe:
#   xgb.XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1,
#                     eval_metric="logloss", random_state=SEED)
# on the UNSCALED training part (trees do not care about scale). The test
# probabilities as p_xgb (hole 5 needs them); accuracy and AUC on the test
# part, next to those of the logistic regression.

model = ...

# Explaining the black box: SHAP

`shap.TreeExplainer` computes exact Shapley values for tree ensembles
(TreeSHAP, Lundberg et al.) - no sampling. The values are in log-odds and add
up, with the base value, to the model's output - the cell checks it.

In [ ]:
explainer = shap.TreeExplainer(model)
explanation = explainer(X_test)
values, base = explanation.values, float(explanation.base_values[0])
print(f"base value {base:.3f} log-odds = P {1 / (1 + np.exp(-base)):.3f}")
print("check - SHAP values add up to the model's log-odds:",
      np.allclose(base + values.sum(1), model.predict(X_test, output_margin=True), atol=1e-4))

### Hole 4 - the global view

In [ ]:
# TODO 4: the GLOBAL view - which features matter most on average over the
#         76 test patients? shap.plots.beeswarm(explanation) shows every
#         patient as a dot per feature; shap.plots.bar(explanation) the mean |SHAP|.

...

### Hole 5 - three patients

In [ ]:
# TODO 5: the LOCAL view - three patients: the most confident "healthy"
#         (lowest p_xgb), the borderline one (p_xgb closest to 0.5) and the
#         most confident "high risk" (highest p_xgb). For each print the
#         probability and the true label, and draw
#         shap.plots.waterfall(explanation[i]).

...

# Do the two explanations agree?

### Hole 6 - coefficients vs. mean |SHAP|

In [ ]:
# TODO 6: do the two explanations agree? A table of |coefficient| of the
#         logistic regression and mean |SHAP| of XGBoost per feature, each
#         column divided by its own maximum, sorted by SHAP, and drawn as bars.

comparison = ...

# Questions

1. Did the accuracy gap justify the black box on these data?
2. For the borderline patient: which feature pushed the prediction up most,
   which one down most? Check that the waterfall adds up.
3. Name a feature on which the two methods disagree. Why might a linear model
   and a tree ensemble use it differently?
4. As a clinician, which explanation would you trust more - the coefficients
   or XGBoost with SHAP - and why?

---

*The numbers are computed by [`heart.py`](https://github.com/tomasvicar/AUI-public/blob/master/lectures/explainability/code/heart.py) of
the lecture. The notebook is built by
[`code/build_notebooks.py`](https://github.com/tomasvicar/AUI/blob/master/labs/explainability/code/build_notebooks.py) - editing the
`.ipynb` by hand gets overwritten.*